In [1]:
import pandas as pd

# Load the original dataset
df = pd.read_csv("teaching_prices_messy.csv")

# Create a separate working copy
clean_df = df.copy()

print("Original dataset:", df.shape)
clean_df.head()

Original dataset: (97, 6)


,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


In [2]:
# Check missing values
print("Missing values:")
print(clean_df.isnull().sum())

# Convert date into datetime format
clean_df["date"] = pd.to_datetime(clean_df["date"])

# Convert prices into numbers
clean_df["price"] = pd.to_numeric(
    clean_df["price"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

print("\nUpdated data types:")
print(clean_df.dtypes)

Missing values:
date          0
item          0
city          0
price         1
unit          0
data_label    0
dtype: int64

Updated data types:
date          datetime64[us]
item                     str
city                     str
price                float64
unit                     str
data_label               str
dtype: object


In [3]:
# Find missing prices
print("Missing prices:")
print(clean_df[clean_df["price"].isnull()][["date", "item", "city", "price"]])

# Find unusually high prices
print("\nSuspicious prices:")
print(clean_df[clean_df["price"] > 500][["date", "item", "city", "price"]])

# Create an audit log
audit = []

audit.append({
    "Issue": "Missing price",
    "Action": "Flagged",
    "Reason": "Correct value could not be verified"
})

audit.append({
    "Issue": "Prices 1250 and 9999",
    "Action": "Flagged",
    "Reason": "Prices are unusually high"
})

# Exclude suspicious prices from numerical analysis
clean_df.loc[clean_df["price"] > 500, "price"] = None

Missing prices:
        date     item    city  price
5 2026-01-01  Pulse C  City 2    NaN

Suspicious prices:
         date         item    city   price
17 2026-01-15     Staple A  City 2  1250.0
30 2026-01-22  Sweetener D  City 1  9999.0


In [4]:
# Correct the inconsistent item name
clean_df.loc[
    (clean_df["item"].str.lower() == "oil b") &
    (clean_df["city"] == "City 1") &
    (clean_df["date"] == pd.Timestamp("2026-02-05")),
    "item"
] = "Staple A"

audit.append({
    "Issue": "Incorrect item name",
    "Action": "Modified",
    "Reason": "Price pattern suggests the correct name is Staple A"
})

# Check and remove duplicate records
duplicates = clean_df.duplicated().sum()
clean_df = clean_df.drop_duplicates()

audit.append({
    "Issue": "Duplicate record",
    "Action": "Removed",
    "Reason": "Exact duplicate observation"
})

print("Duplicates found:", duplicates)
print("Duplicates removed:", duplicates)

Duplicates found: 1
Duplicates removed: 1


In [5]:
# Save the cleaned dataset
clean_df.to_csv("teaching_prices_clean_working.csv", index=False)

# Save the audit log
audit_df = pd.DataFrame(audit)
audit_df.to_csv("cleaning_audit.csv", index=False)

# Display final results
print("Cleaning completed")
print("Original rows:", len(df))
print("Cleaned rows:", len(clean_df))
print("Missing prices:", clean_df["price"].isnull().sum())
print("Remaining duplicates:", clean_df.duplicated().sum())

audit_df

Cleaning completed
Original rows: 97
Cleaned rows: 96
Missing prices: 3
Remaining duplicates: 0


,Issue,Action,Reason
0,Missing price,Flagged,Correct value could not be verified
1,Prices 1250 and 9999,Flagged,Prices are unusually high
2,Incorrect item name,Modified,Price pattern suggests the correct name is Sta...
3,Duplicate record,Removed,Exact duplicate observation
